In [2]:
import cv2
import os
import glob
import numpy as np


# FINAL SEGMENTATION PIPELINE
# Pothole             = Red
# Alligator Cracking  = Green
# Center Line Removal = Yellow
# No text labels


input_dir = r"D:\Downloads\IP_Project\Enhanced_frames"
output_dir = r"D:\Downloads\IP_Project\Segmentation"

os.makedirs(output_dir, exist_ok=True)

# -------OUTPUT FOLDERS-------

roi_dir        = os.path.join(output_dir, "ROI_masks")
lane_dir       = os.path.join(output_dir, "Lane_masks")
dark_dir       = os.path.join(output_dir, "Dark_damage_masks")
pothole_dir    = os.path.join(output_dir, "Pothole_masks")
alligator_dir  = os.path.join(output_dir, "Alligator_masks")
combined_dir   = os.path.join(output_dir, "Combined_masks")
seg_dir        = os.path.join(output_dir, "Segmented_frames")
comparison_dir = os.path.join(output_dir, "Comparison_results")
report_dir     = os.path.join(output_dir, "Reports")

folders = [
    roi_dir,
    lane_dir,
    dark_dir,
    pothole_dir,
    alligator_dir,
    combined_dir,
    seg_dir,
    comparison_dir,
    report_dir
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

# -------READ ENHANCED FRAMES-------

frames = sorted(glob.glob(os.path.join(input_dir, "*.jpg")))

print("Total enhanced frames found:", len(frames))

if len(frames) == 0:
    print("No frames found. Check Enhanced_frames folder path.")
    exit()

# -------PROCESS EACH FRAME-------

for path in frames:

    img = cv2.imread(path)

    if img is None:
        print("Skipped:", path)
        continue

    file_name = os.path.basename(path)
    output = img.copy()

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    # -------1. ROAD ROI MASK-------
    

    roi_mask = np.zeros_like(gray)

    roi_points = np.array([[
        (int(w * 0.05), int(h * 1.00)),
        (int(w * 0.08), int(h * 0.45)),
        (int(w * 0.30), int(h * 0.25)),
        (int(w * 0.70), int(h * 0.25)),
        (int(w * 0.92), int(h * 0.45)),
        (int(w * 0.95), int(h * 1.00))
    ]], dtype=np.int32)

    cv2.fillPoly(roi_mask, roi_points, 255)

    # ROI visualization
    roi_output = img.copy()
    roi_output[roi_mask == 255] = cv2.addWeighted(
        roi_output[roi_mask == 255],
        0.5,
        np.full_like(roi_output[roi_mask == 255], (255, 0, 0)),
        0.5,
        0
    )

    # -------2. BLUR IMAGE-------

    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # -------3. CENTER LINE REMOVAL MASK - YELLOW-------

    center_line_area = np.zeros_like(gray)

    center_line_area[:, int(w * 0.38):int(w * 0.62)] = 255
    center_line_area = cv2.bitwise_and(center_line_area, roi_mask)

    _, lane_mask = cv2.threshold(
        blur,
        120,
        255,
        cv2.THRESH_BINARY
    )

    lane_mask = cv2.bitwise_and(lane_mask, center_line_area)

    lane_kernel = np.ones((9, 9), np.uint8)
    lane_mask = cv2.morphologyEx(lane_mask, cv2.MORPH_CLOSE, lane_kernel)
    lane_mask = cv2.morphologyEx(lane_mask, cv2.MORPH_OPEN, lane_kernel)

    # -------4. DARK DAMAGE MASK-------

    dark_mask = cv2.adaptiveThreshold(
        blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        41,
        8
    )

    dark_mask = cv2.bitwise_and(dark_mask, roi_mask)

    small_kernel = np.ones((3, 3), np.uint8)
    dark_mask = cv2.morphologyEx(dark_mask, cv2.MORPH_OPEN, small_kernel)

    # -------5. POTHOLE MASK - RED-------

    pothole_mask = np.zeros_like(gray)

    contours, _ = cv2.findContours(
        dark_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours:

        area = cv2.contourArea(cnt)

        if area < 700:
            continue

        perimeter = cv2.arcLength(cnt, True)

        if perimeter == 0:
            continue

        x, y, cw, ch = cv2.boundingRect(cnt)
        aspect_ratio = cw / float(ch)
        circularity = (4 * np.pi * area) / (perimeter * perimeter)

        if 0.4 < aspect_ratio < 2.8 and circularity > 0.12:
            cv2.drawContours(pothole_mask, [cnt], -1, 255, -1)

    # -------6. ALLIGATOR CRACK MASK - GREEN-------

    alligator_mask = cv2.subtract(dark_mask, pothole_mask)

    crack_kernel = np.ones((3, 3), np.uint8)
    alligator_mask = cv2.morphologyEx(
        alligator_mask,
        cv2.MORPH_CLOSE,
        crack_kernel
    )

    # -------7. DRAW POTHOLES - RED DOTS-------

    pothole_contours, _ = cv2.findContours(
        pothole_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in pothole_contours:

        area = cv2.contourArea(cnt)

        if area < 700:
            continue

        mask_temp = np.zeros_like(gray)
        cv2.drawContours(mask_temp, [cnt], -1, 255, -1)

        ys, xs = np.where(mask_temp == 255)

        for i in range(0, len(xs), 10):
            cv2.circle(output, (xs[i], ys[i]), 2, (0, 0, 255), -1)

    # -------8. DRAW ALLIGATOR CRACKS - GREEN LINES-------

    alligator_contours, _ = cv2.findContours(
        alligator_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in alligator_contours:

        area = cv2.contourArea(cnt)

        if area < 80:
            continue

        x, y, cw, ch = cv2.boundingRect(cnt)

        if cw > 12 and ch > 12:
            cv2.drawContours(output, [cnt], -1, (0, 255, 0), 2)

    # -------9. DRAW CENTER LINE REMOVAL - YELLOW-------

    lane_contours, _ = cv2.findContours(
        lane_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in lane_contours:

        area = cv2.contourArea(cnt)

        if area < 200:
            continue

        x, y, cw, ch = cv2.boundingRect(cnt)
        aspect_ratio = max(cw, ch) / max(1, min(cw, ch))

        if aspect_ratio > 1.2:
            cv2.drawContours(output, [cnt], -1, (0, 255, 255), 2)

    # -------10. COMBINED MASK-------

    combined_mask = cv2.bitwise_or(pothole_mask, alligator_mask)
    combined_mask = cv2.bitwise_or(combined_mask, lane_mask)

    # -------11. COMPARISON IMAGE-------

    comparison = np.hstack((img, output))

    # -------12. SAVE OUTPUTS-------

    cv2.imwrite(os.path.join(roi_dir, "roi_" + file_name), roi_mask)
    cv2.imwrite(os.path.join(lane_dir, "lane_" + file_name), lane_mask)
    cv2.imwrite(os.path.join(dark_dir, "dark_" + file_name), dark_mask)
    cv2.imwrite(os.path.join(pothole_dir, "pothole_" + file_name), pothole_mask)
    cv2.imwrite(os.path.join(alligator_dir, "alligator_" + file_name), alligator_mask)
    cv2.imwrite(os.path.join(combined_dir, "combined_" + file_name), combined_mask)
    cv2.imwrite(os.path.join(seg_dir, "seg_" + file_name), output)
    cv2.imwrite(os.path.join(comparison_dir, "compare_" + file_name), comparison)


# REPORT TEXT FILE

report_path = os.path.join(report_dir, "segmentation_report.txt")

with open(report_path, "w") as report:
    report.write("Final Segmentation Pipeline Report\n")
    report.write("=================================\n\n")
    report.write("Input Folder: " + input_dir + "\n")
    report.write("Output Folder: " + output_dir + "\n\n")
    report.write("Detected Classes:\n")
    report.write("1. Pothole - Red dots\n")
    report.write("2. Alligator Cracking - Green contours\n")
    report.write("3. Center Line Removal - Yellow contours\n\n")
    report.write("Created Output Folders:\n")
    report.write("1. ROI_masks\n")
    report.write("2. Lane_masks\n")
    report.write("3. Dark_damage_masks\n")
    report.write("4. Pothole_masks\n")
    report.write("5. Alligator_masks\n")
    report.write("6. Combined_masks\n")
    report.write("7. Segmented_frames\n")
    report.write("8. Comparison_results\n")
    report.write("9. Reports\n\n")
    report.write("Total Frames Processed: " + str(len(frames)) + "\n")

print("Segmentation completed successfully.")
print("ROI masks saved in:", roi_dir)
print("Lane masks saved in:", lane_dir)
print("Dark masks saved in:", dark_dir)
print("Pothole masks saved in:", pothole_dir)
print("Alligator masks saved in:", alligator_dir)
print("Combined masks saved in:", combined_dir)
print("Segmented frames saved in:", seg_dir)
print("Comparison results saved in:", comparison_dir)
print("Report saved in:", report_path)

Total enhanced frames found: 453
Segmentation completed successfully.
ROI masks saved in: D:\Downloads\IP_Project\Segmentation\ROI_masks
Lane masks saved in: D:\Downloads\IP_Project\Segmentation\Lane_masks
Dark masks saved in: D:\Downloads\IP_Project\Segmentation\Dark_damage_masks
Pothole masks saved in: D:\Downloads\IP_Project\Segmentation\Pothole_masks
Alligator masks saved in: D:\Downloads\IP_Project\Segmentation\Alligator_masks
Combined masks saved in: D:\Downloads\IP_Project\Segmentation\Combined_masks
Segmented frames saved in: D:\Downloads\IP_Project\Segmentation\Segmented_frames
Comparison results saved in: D:\Downloads\IP_Project\Segmentation\Comparison_results
Report saved in: D:\Downloads\IP_Project\Segmentation\Reports\segmentation_report.txt
